In [236]:
import torch
from math import sqrt

In [97]:
with open(
    "Obras_completas_ilustradas_Federico_Garcia_Lorca_djvu.txt",
    "r",
    encoding="utf-8"
) as f:

    text = f.read()


In [98]:
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: [itos[i] for i in l]

In [99]:
text[:8]
encoded_text = torch.tensor(encode(text), dtype = torch.long)

In [255]:
train_size = int(len(text)*0.9)
train_data = encoded_text[:train_size]
test_data = encoded_text[train_size:]
block_size = 8
batch_size = 1
torch.manual_seed(0)

def get_batch(split):
    # generate a small batch of data of inputs x and targets y

    ix = torch.randint(len(split) - block_size, (batch_size,))
    x = torch.stack([split[i:i+block_size] for i in ix])
    y = torch.stack([split[i+1:i+block_size+1] for i in ix])
    return x, y

In [706]:
x, y = get_batch(train_data)

In [397]:
vocab_size = len(chars)
d_model = 10
embedding = torch.nn.Embedding(vocab_size, d_model)

In [398]:
vocab_size

110

In [710]:

class Head(torch.nn.Module):

    def __init__(self, block_size, d_model, d_v, d_k):
        super().__init__()
        self.d_k = d_k
        self.W_q = torch.nn.Linear(d_model, d_k)
        self.W_k = torch.nn.Linear(d_model, d_k)
        self.W_v = torch.nn.Linear(d_model, d_v)
        self.register_buffer("causal_mask", self.ins_causal_mask(block_size))


    def forward(self, X_e):
        B, T, d_model = X_e.shape
        Q = self.W_q(X_e) #(B,T,d_k)
        K = self.W_k(X_e) #(B,T,d_k)
        V = self.W_v(X_e) #(B,T,d_v)
        QK = Q @ K.transpose(1,2)
        QK_scale = QK/math.sqrt(self.d_k) #(B,T,T)
        QK_scale += self.causal_mask[:T, :T]
        QV_softmax = torch.nn.functional.softmax(QK_scale, dim = 2) #(B,T,T)
        added_weights = QV_softmax @ V #(B, T, d_v)
        return added_weights


    @staticmethod
    def ins_causal_mask(block_size):
        mask = torch.triu(torch.ones(block_size, block_size), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask


class MLP(torch.nn.Module):

    def __init__(self, d_model):
            
        super().__init__()
        self.W_ff1 = torch.nn.Linear(d_model, 4*d_model)
        self.W_ff2 = torch.nn.Linear(4*d_model, d_model)

    def forward(self, X):

        linear_1 = self.W_ff1(X)
        non_linearity = torch.nn.functional.gelu(linear_1)
        linear_2 = self.W_ff2(non_linearity)
        return linear_2

        

class MultiHead(torch.nn.Module):
    
    def __init__(self, block_size, d_model, n_heads, d_k):
        super().__init__()
        assert d_model % n_heads == 0
        d_v = d_model // n_heads
        d_k = d_k if d_k is not None else d_v
        self.heads = torch.nn.ModuleList([Head(block_size, d_model, d_v, d_k) for _ in range(n_heads)])
        self.W_o = torch.nn.Linear(d_model, d_model)

    def forward(self, X_e):

        added_weights = [head(X_e) for head in self.heads]
        concat = torch.cat(added_weights, dim = -1)
        return self.W_o(concat)

class Block(torch.nn.Module):
     
    def __init__(self, block_size, d_model, n_heads, d_k):
        super().__init__()
        self.att = MultiHead(block_size, d_model, n_heads, d_k)
        self.ln_1 = torch.nn.LayerNorm(d_model)
        self.mlp = MLP(d_model)
        self.ln_2 = torch.nn.LayerNorm(d_model)

    def forward(self, X):

        X = X + self.att(self.ln_1(X))
        X = X + self.mlp(self.ln_2(X))
        return X


class decoder_only_transformer(torch.nn.Module):

    def __init__(self, vocab_size, block_size, d_model, n_heads = 1, d_k = None, n_layers = 1):

        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, d_model) 
        self.blocks = torch.nn.ModuleList([Block(block_size, d_model, n_heads, d_k) for _ in range(n_layers)])
        self.ln_f = torch.nn.LayerNorm(d_model)
        self.linear = torch.nn.Linear(d_model, vocab_size)
        self.register_buffer("pos_encoding", self.sinusoidal_pe(block_size, d_model))
        self.loss = torch.nn.CrossEntropyLoss()

    
    @staticmethod
    def sinusoidal_pe(block_size, d_model):
        pe = torch.zeros(block_size, d_model)
        for pos in range(block_size):
            for i in range(0, d_model, 2):
                denom = 10000 ** (i / d_model)
                pe[pos, i]     = math.sin(pos / denom)
                pe[pos, i + 1] = math.cos(pos / denom)
        return pe

    
    
    def forward(self, X, Y = None):

        B, T = X.shape
        X_e = self.embedding(X) + self.pos_encoding[:T] #(B,T,d_model)
        for block in self.blocks:
            X_e = block(X_e)
        X_e = self.ln_f(X_e) #(B,T,d_model)
        logits = self.linear(X_e) #(B,T,vocab_size)
        loss = None
        if Y is not None:
            loss = self.loss(logits.reshape(B*T,-1), Y.reshape(B*T,))
        
        return logits, loss



In [712]:
vocab_size = len(chars)          # tamaño de tu vocabulario (65 en tinyshakespeare)
block_size = 8                   # longitud máxima de contexto

modelo = decoder_only_transformer(
    vocab_size=vocab_size,
    block_size=block_size,
    d_model=32,
    n_heads=4,
    n_layers=2,
)
logits, loss = modelo(x, y);  print(logits.shape, loss.item(), math.log(vocab_size))
logits, loss = modelo(x);     print(logits.shape, loss)      # loss → None

torch.Size([1, 8, 110]) 4.8519487380981445 4.700480365792417
torch.Size([1, 8, 110]) None


In [702]:
import torch

# Model instance
vocab_size = len(chars)
print(vocab_size)
d_model = 10
block_size = 8
d_k = 20
n_heads = 5

modelo = decoder_only_transformer(vocab_size, block_size, d_model, n_heads = 1, d_k = 16)
modelo.eval()

# Without gradients
with torch.no_grad():
    logits = modelo(first_batch[0], first_batch[1])  # Ejecuta el forward()

    clases_predichas = torch.argmax(logits, dim=-1)

    # Opción B: Obtener las probabilidades (0.0 a 1.0) para cada clase
    probabilidades = torch.softmax(logits, dim=-1)

# Imprimir resultados
print("Logits raw:\n", logits)
print("Classes predichas:\n", decode(clases_predichas))
print("Probabilidades:\n", probabilidades)

110
torch.Size([1, 8]) torch.Size([1, 8, 110]) torch.Size([1, 8])


/var/folders/cd/dy43bkgs6hqc2zhw43gtt70c0000gn/T/ipykernel_74326/2355525279.py:116: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss = self.loss(logits.reshape(B*T,-1), torch.tensor(Y.reshape(B*T,)))


TypeError: argmax(): argument 'input' (position 1) must be Tensor, not tuple

<method-wrapper '__repr__' of builtin_function_or_method object at 0x14008f830>

In [420]:
clases_predichas.tolist()[0]

[51, 85, 55, 48, 58, 58, 53, 62]

In [ ]:
my_first_model = decoder_only_transformer()
my_first_model(first_batch[0])

In [421]:
decode(clases_predichas.tolist()[0])

['X', '¡', '\\', 'U', 'a', 'a', 'Z', 'e']

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(first_batch[0][0], first_batch[1][0])
print(f'Total loss for this batch: {loss.item()}')

In [ ]:
vocab_size = len(chars)
embedding_dim = 10
embedding = torch.nn.Embedding(vocab_size, embedding_dim)